# Visualização de Espécies de Árvores

Este notebook gera visualizações de espécies de árvores para dois conjuntos de dados: **Amazon Data** e **Bioflore Data**.

Para cada conjunto de dados, geramos um grid onde:
- Cada linha representa uma espécie.
- Cada linha contém 4 amostras daquela espécie.
- As árvores são mostradas em destaque (foreground) com fundo preto.
- Os recortes são quadrados e centralizados para cobrir a copa.
- O título da espécie é exibido centralizado sobre a linha.

## Configurações e Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box, shape
import matplotlib.pyplot as plt
from skimage import measure
from rasterio.windows import Window
from shapely.affinity import translate
import cv2

# Configurações de plotagem
plt.rcParams['figure.figsize'] = (16, 20)
plt.rcParams['image.interpolation'] = 'antialiased'

## Funções Auxiliares

In [ ]:
# def plot_species_grid_wrapped(species_samples, dataset_name):
#     """
#     Plota um grid de visualização das espécies com wrap.
#     - 1 coluna por espécie
#     - 2 amostras por espécie (empilhadas verticalmente na mesma coluna)
#     - Metade das espécies nas duas primeiras linhas
#     - Outra metade nas duas linhas de baixo
    
#     Layout:
#     Linhas 0-1: primeira metade das espécies (cada espécie em 1 coluna com 2 amostras)
#     Linhas 2-3: segunda metade das espécies (cada espécie em 1 coluna com 2 amostras)
    
#     species_samples: dict onde chaves são nomes das espécies e valores são listas de arrays de imagem (RGB).
#     dataset_name: Nome do dataset para o título.
#     """
#     unique_species = sorted(list(species_samples.keys()))
#     n_species = len(unique_species)
#     samples_per_species = 2
    
#     if n_species == 0:
#         print(f"Nenhuma espécie para plotar em {dataset_name}.")
#         return

#     # Dividir espécies em duas metades
#     half_point = (n_species + 1) // 2  # Arredonda para cima
#     first_half = unique_species[:half_point]
#     second_half = unique_species[half_point:]
    
#     n_first_half = len(first_half)
#     n_second_half = len(second_half)
    
#     # Grid: 4 linhas × max(n_first_half, n_second_half) colunas (1 coluna por espécie)
#     n_cols = max(n_first_half, n_second_half)
#     n_rows = 4
    
#     # Usar GridSpec para controlar espaçamento vertical diferencial
#     from matplotlib.gridspec import GridSpec
    
#     # Criar figura e GridSpec com heights diferentes
#     # heights: [linha0, espaço_pequeno, linha1, espaço_grande, linha2, espaço_pequeno, linha3]
#     fig = plt.figure(figsize=(n_cols * 2.2, 6.5))
#     gs = GridSpec(4, n_cols, figure=fig, 
#                   height_ratios=[1, 1, 1, 1],  # Mesma altura para cada linha
#                   hspace=0.05,  # Espaçamento pequeno entre linhas adjacentes
#                   wspace=0.5)    # Espaçamento horizontal entre colunas
    
#     # Criar array de axes
#     axes = []
#     for i in range(n_rows):
#         row_axes = []
#         for j in range(n_cols):
#             ax = fig.add_subplot(gs[i, j])
#             row_axes.append(ax)
#         axes.append(row_axes)
#     axes = np.array(axes)
    
#     # Adicionar espaço extra entre as duas metades (entre linha 1 e 2)
#     # Ajustar posição das linhas 2 e 3 para baixo
#     for j in range(n_cols):
#         # Mover linhas 2 e 3 para baixo criando mais espaço
#         bbox_0 = axes[0, j].get_position()
#         bbox_1 = axes[1, j].get_position()
#         bbox_2 = axes[2, j].get_position()
#         bbox_3 = axes[3, j].get_position()
        
#         # Calcular nova posição: manter linhas 0-1 próximas, aumentar espaço antes da linha 2
#         height_01 = bbox_1.y1 - bbox_0.y0  # Altura total das linhas 0-1
#         spacing_between_groups = 0.15  # Espaço extra entre grupos (15% da altura total)
        
#         # Reposicionar linhas 2 e 3
#         new_y0_2 = bbox_1.y0 - spacing_between_groups
#         new_height = bbox_2.height
        
#         axes[2, j].set_position([bbox_2.x0, new_y0_2 - new_height, bbox_2.width, new_height])
#         axes[3, j].set_position([bbox_3.x0, new_y0_2 - 2*new_height - 0.05, bbox_3.width, new_height])
    
#     def break_long_title(title, max_length=12):
#         """Quebra títulos longos em múltiplas linhas."""
#         words = title.split()
#         if len(title) <= max_length:
#             return title
        
#         lines = []
#         current_line = []
#         current_length = 0
        
#         for word in words:
#             if current_length + len(word) + 1 <= max_length:
#                 current_line.append(word)
#                 current_length += len(word) + 1
#             else:
#                 if current_line:
#                     lines.append(' '.join(current_line))
#                 current_line = [word]
#                 current_length = len(word)
        
#         if current_line:
#             lines.append(' '.join(current_line))
        
#         return chr(10).join(lines)
    
#     # Primeira metade: linhas 0 e 1
#     for j, species in enumerate(first_half):
#         samples = species_samples[species]
        
#         # Se não tiver 2 amostras, preencher com imagens vazias
#         while len(samples) < samples_per_species:
#             samples.append(np.zeros((100, 100, 3), dtype=np.uint8))
        
#         # Primeira amostra na linha 0, coluna j
#         axes[0, j].imshow(samples[0])
#         axes[0, j].axis('off')
        
#         # Segunda amostra na linha 1, mesma coluna j
#         axes[1, j].imshow(samples[1])
#         axes[1, j].axis('off')
        
#         # Título da espécie quebrado se necessário
#         title = break_long_title(species)
#         axes[0, j].text(0.5, 1.08, title, transform=axes[0, j].transAxes, 
#                         ha='center', va='bottom', fontsize=12)
    
#     # Segunda metade: linhas 2 e 3
#     for j, species in enumerate(second_half):
#         samples = species_samples[species]
        
#         # Se não tiver 2 amostras, preencher com imagens vazias
#         while len(samples) < samples_per_species:
#             samples.append(np.zeros((100, 100, 3), dtype=np.uint8))
        
#         # Primeira amostra na linha 2, coluna j
#         axes[2, j].imshow(samples[0])
#         axes[2, j].axis('off')
        
#         # Segunda amostra na linha 3, mesma coluna j
#         axes[3, j].imshow(samples[1])
#         axes[3, j].axis('off')
        
#         # Título da espécie quebrado se necessário
#         title = break_long_title(species)
#         axes[2, j].text(0.5, 1.08, title, transform=axes[2, j].transAxes, 
#                         ha='center', va='bottom', fontsize=12)
    
#     # Desabilitar apenas os eixos vazios (colunas não usadas)
#     for i in range(n_rows):
#         for j in range(n_cols):
#             # Se está nas linhas 0-1 mas além da primeira metade
#             if i < 2 and j >= n_first_half:
#                 axes[i, j].axis('off')
#             # Se está nas linhas 2-3 mas além da segunda metade
#             elif i >= 2 and j >= n_second_half:
#                 axes[i, j].axis('off')

#     plt.tight_layout()
#     plt.savefig(f'{dataset_name}_species_grid_wrapped.pdf', bbox_inches='tight', dpi=100)
#     plt.show()

def plot_species_grid_3percol(species_samples, dataset_name):
    """
    Plota um grid de visualização das espécies com 3 espécies por coluna.
    - 3 espécies por coluna (empilhadas verticalmente)
    - 2 amostras por espécie
    - 6 linhas no total (2 linhas por espécie)
    
    Layout:
    Linhas 0-1: primeira espécie da coluna (2 amostras)
    Linhas 2-3: segunda espécie da coluna (2 amostras)
    Linhas 4-5: terceira espécie da coluna (2 amostras)
    
    species_samples: dict onde chaves são nomes das espécies e valores são listas de arrays de imagem (RGB).
    dataset_name: Nome do dataset para o título.
    """
    unique_species = sorted(list(species_samples.keys()))
    n_species = len(unique_species)
    samples_per_species = 2
    species_per_col = 3
    
    if n_species == 0:
        print(f"Nenhuma espécie para plotar em {dataset_name}.")
        return

    # Calcular número de colunas necessárias (3 espécies por coluna)
    n_cols = (n_species + species_per_col - 1) // species_per_col  # Arredonda para cima
    n_rows = 6  # 2 linhas por espécie × 3 espécies
    
    # Usar GridSpec para controlar espaçamento
    from matplotlib.gridspec import GridSpec
    
    # Criar figura e GridSpec
    fig = plt.figure(figsize=(n_cols * 2.2, 12))
    gs = GridSpec(6, n_cols, figure=fig, 
                  height_ratios=[1, 1, 1, 1, 1, 1],  # Mesma altura para cada linha
                  hspace=0.05,  # Espaçamento pequeno entre linhas adjacentes
                  wspace=0.5)    # Espaçamento horizontal entre colunas
    
    # Criar array de axes
    axes = []
    for i in range(n_rows):
        row_axes = []
        for j in range(n_cols):
            ax = fig.add_subplot(gs[i, j])
            row_axes.append(ax)
        axes.append(row_axes)
    axes = np.array(axes)
    
    # Ajustar espaçamento vertical entre espécies (mas manter intra-espécie)
    # Criar mais espaço entre linhas 1-2 (entre primeira e segunda espécie)
    # e entre linhas 3-4 (entre segunda e terceira espécie)
    for j in range(n_cols):
        # Obter posições atuais
        bbox_0 = axes[0, j].get_position()
        bbox_1 = axes[1, j].get_position()
        bbox_2 = axes[2, j].get_position()
        bbox_3 = axes[3, j].get_position()
        bbox_4 = axes[4, j].get_position()
        bbox_5 = axes[5, j].get_position()
        
        # Calcular altura de um grupo de espécie (2 linhas)
        species_height = bbox_1.y1 - bbox_0.y0
        
        # Espaço extra entre espécies (para caber o título)
        spacing_between_species = 0.12  # 12% da altura de uma espécie
        # Distância fixa entre as duas amostras da mesma espécie
        intra_species_spacing = 0.05
        
        # Reposicionar segunda espécie (linhas 2-3) para baixo
        # Posição da primeira amostra da segunda espécie
        new_y1_2 = bbox_1.y0 - spacing_between_species
        axes[2, j].set_position([bbox_2.x0, new_y1_2 - bbox_2.height, bbox_2.width, bbox_2.height])
        # Segunda amostra: manter distância fixa da primeira
        axes[3, j].set_position([bbox_3.x0, new_y1_2 - bbox_2.height - intra_species_spacing - bbox_3.height, bbox_3.width, bbox_3.height])
        
        # Reposicionar terceira espécie (linhas 4-5) para baixo
        # Posição da primeira amostra da terceira espécie
        new_y1_4 = axes[3, j].get_position().y0 - spacing_between_species
        axes[4, j].set_position([bbox_4.x0, new_y1_4 - bbox_4.height, bbox_4.width, bbox_4.height])
        # Segunda amostra: manter distância fixa da primeira
        axes[5, j].set_position([bbox_5.x0, new_y1_4 - bbox_4.height - intra_species_spacing - bbox_5.height, bbox_5.width, bbox_5.height])
    
    def break_long_title(title, max_length=12):
        """Quebra títulos longos em múltiplas linhas."""
        words = title.split()
        if len(title) <= max_length:
            return title
        
        lines = []
        current_line = []
        current_length = 0
        
        for word in words:
            if current_length + len(word) + 1 <= max_length:
                current_line.append(word)
                current_length += len(word) + 1
            else:
                if current_line:
                    lines.append(' '.join(current_line))
                current_line = [word]
                current_length = len(word)
        
        if current_line:
            lines.append(' '.join(current_line))
        
        return chr(10).join(lines)
    
    # Preencher o grid: 3 espécies por coluna
    species_idx = 0
    for j in range(n_cols):
        for species_offset in range(species_per_col):
            if species_idx >= n_species:
                # Preencher com eixos vazios se não houver mais espécies
                for i in range(species_offset * 2, (species_offset + 1) * 2):
                    axes[i, j].axis('off')
                continue
            
            species = unique_species[species_idx]
            samples = species_samples[species]
            species_idx += 1
            
            # Se não tiver 2 amostras, preencher com imagens vazias
            while len(samples) < samples_per_species:
                samples.append(np.zeros((100, 100, 3), dtype=np.uint8))
            
            # Linhas para esta espécie: species_offset * 2 e species_offset * 2 + 1
            row_start = species_offset * 2
            
            # Primeira amostra
            axes[row_start, j].imshow(samples[0])
            axes[row_start, j].axis('off')
            
            # Segunda amostra
            axes[row_start + 1, j].imshow(samples[1])
            axes[row_start + 1, j].axis('off')
            
            # Título da espécie quebrado se necessário
            title = break_long_title(species)
            axes[row_start, j].text(0.5, 1.08, title, transform=axes[row_start, j].transAxes, 
                            ha='center', va='bottom', fontsize=12)

    plt.tight_layout()
    plt.savefig(f'{dataset_name}_species_grid_3percol.pdf', bbox_inches='tight', dpi=100)
    plt.show()

## 1. Amazon Data

Arquivos utilizados:
- Imagem: `amazon_data/amazon_input_data/orthoimage/orthoimage.tif`
- Segmentação: `amazon_data/amazon_input_data/segmentation/ITC_CLASS.tif`
- IDs: `amazon_data/amazon_input_data/id_trees.csv`

In [ ]:
# Paths
amazon_img_path = '/home/luizluz/Documentos/multi-task-fcn/amazon_data/amazon_input_data/orthoimage/orthoimage.tif'
amazon_seg_path = '/home/luizluz/Documentos/multi-task-fcn/amazon_data/amazon_input_data/segmentation/ITC_CLASS.tif'
amazon_csv_path = '/home/luizluz/Documentos/multi-task-fcn/amazon_data/amazon_input_data/id_trees.csv'

# Load species mapping
df_amazon = pd.read_csv(amazon_csv_path)
unique_species_amazon = df_amazon['tree_name'].unique()
print(f"Espécies encontradas no CSV: {unique_species_amazon}")

# Prepara um dicionário para guardar as amostras
amazon_samples = {sp: [] for sp in unique_species_amazon}

# Função para verificar sobreposição de patches
def patches_overlap(center1_r, center1_c, size1, center2_r, center2_c, size2):
    """Verifica se dois patches se sobrepõem"""
    half1 = size1 // 2
    half2 = size2 // 2
    
    # Bounds do primeiro patch
    minr1 = center1_r - half1
    maxr1 = center1_r + half1
    minc1 = center1_c - half1
    maxc1 = center1_c + half1
    
    # Bounds do segundo patch
    minr2 = center2_r - half2
    maxr2 = center2_r + half2
    minc2 = center2_c - half2
    maxc2 = center2_c + half2
    
    # Verificar sobreposição: se não há gap entre os patches, eles se sobrepõem
    return not (maxr1 < minr2 or maxr2 < minr1 or maxc1 < minc2 or maxc2 < minc1)

# Função para extrair amostras
def extract_amazon_samples(img_path, seg_path, df, samples_dict, max_samples=4):
    print("Lendo máscara de classes...")
    with rasterio.open(seg_path) as src_seg:
        seg_data = src_seg.read(1)
        profile = src_seg.profile

    print("Lendo imagem RGB...")
    with rasterio.open(img_path) as src_img:
        # Loop por espécie
        for species in samples_dict.keys():
            species_id = df[df['tree_name'] == species]['label_num'].iloc[0]
            print(f"Processando: {species} (ID: {species_id})")
            
            # Mascara binária para o ID da espécie
            mask_species = (seg_data == species_id)
            
            # Encontrar regiões (árvores individuais)
            labeled_mask = measure.label(mask_species)
            regions = measure.regionprops(labeled_mask)
            
            # Ordenar por área (maiores primeiro)
            regions = sorted(regions, key=lambda x: x.area, reverse=True)
            
            count = 0
            # Lista para guardar os patches já processados (center_r, center_c, size)
            processed_patches = []
            
            for region in regions:
                if count >= max_samples:
                    break
                
                # Bounding box da árvore (min_row, min_col, max_row, max_col)
                minr, minc, maxr, maxc = region.bbox
                
                # Calcular tamanho dinâmico baseado na árvore com margem de 30%
                height = maxr - minr
                width = maxc - minc
                
                # Tamanho do recorte: maior dimensão + 30% de margem, mínimo 100px
                size = max(100, int(max(height, width) * 1.05))
                
                # Garantir que seja par para facilitar centralização
                if size % 2 != 0:
                    size += 1
                
                center_r = minr + height // 2
                center_c = minc + width // 2
                half_size = size // 2
                
                # Verificar se a janela está dentro dos limites
                if (center_r - half_size < 0 or center_r + half_size > seg_data.shape[0] or
                    center_c - half_size < 0 or center_c + half_size > seg_data.shape[1]):
                    continue
                
                # Verificar se este patch sobrepõe com algum já processado
                overlaps = False
                for prev_center_r, prev_center_c, prev_size in processed_patches:
                    if patches_overlap(center_r, center_c, size, prev_center_r, prev_center_c, prev_size):
                        overlaps = True
                        break
                
                if overlaps:
                    continue  # Pular esta árvore se sobrepõe com uma já processada
                
                # Janela de leitura
                window = Window(center_c - half_size, center_r - half_size, size, size)
                
                try:
                    rgb_crop = src_img.read(window=window)
                    
                    # Verificar se leu corretamente
                    if rgb_crop.shape[1] != size or rgb_crop.shape[2] != size:
                        continue
                    
                    # Recortar segmentação usando a mesma janela
                    # Window usa (col_off, row_off), então precisamos converter
                    # seg_data é (rows, cols), então usamos [row_start:row_end, col_start:col_end]
                    row_start = center_r - half_size
                    row_end = center_r + half_size
                    col_start = center_c - half_size
                    col_end = center_c + half_size
                    seg_crop = seg_data[row_start:row_end, col_start:col_end]
                    
                    # Recortar também o labeled_mask para identificar apenas esta árvore específica
                    labeled_crop = labeled_mask[row_start:row_end, col_start:col_end]
                    
                    # Criar máscara binária apenas para esta árvore específica (não todas da espécie)
                    mask_crop = (labeled_crop == region.label)
                    
                    # Aplicar máscara no RGB (fundo preto)
                    rgb_masked = rgb_crop.copy()
                    for b in range(3):
                        rgb_masked[b][~mask_crop] = 0
                    
                    # Transpor para (H, W, 3) para plotar
                    img_show = np.transpose(rgb_masked, (1, 2, 0))
                    
                    # Adicionar à lista
                    samples_dict[species].append(img_show)
                    # Registrar este patch como processado
                    processed_patches.append((center_r, center_c, size))
                    count += 1
                    
                except Exception as e:
                    print(f"Erro ao recortar: {e}")
                    continue

# Verificar se as amostras já foram carregadas
if not any(len(samples) > 0 for samples in amazon_samples.values()):
    print("Extraindo amostras do Amazon Data...")
    extract_amazon_samples(amazon_img_path, amazon_seg_path, df_amazon, amazon_samples)
else:
    print("Amostras do Amazon Data já foram carregadas. Pulando extração.")


In [ ]:
def plot_species_grid_3percol(species_samples, dataset_name):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec

    unique_species = sorted(list(species_samples.keys()))
    n_species = len(unique_species)
    samples_per_species = 2
    species_per_col = 3

    if n_species == 0:
        print(f"Nenhuma espécie para plotar em {dataset_name}.")
        return

    n_cols = (n_species + species_per_col - 1) // species_per_col

    # 2 linhas por espécie + 2 linhas spacer entre espécies => 8 linhas
    n_rows = 8
    # linhas: 0,1 (esp1) | 2 spacer | 3,4 (esp2) | 5 spacer | 6,7 (esp3)
    spacer_rows = {2, 5}

    intra_hspace = 0.04
    inter_gap = 0.22  # “altura” do spacer

    fig = plt.figure(figsize=(n_cols * 2.2, 12))
    gs = GridSpec(
        n_rows, n_cols, figure=fig,
        height_ratios=[1, 1, inter_gap, 1, 1, inter_gap, 1, 1],
        hspace=intra_hspace,
        wspace=0.5
    )

    axes = np.empty((n_rows, n_cols), dtype=object)
    for i in range(n_rows):
        for j in range(n_cols):
            ax = fig.add_subplot(gs[i, j])
            axes[i, j] = ax
            ax.axis("off")

    def break_long_title(title, max_length=12):
        words = title.split()
        if len(title) <= max_length:
            return title
        lines, cur, cur_len = [], [], 0
        for w in words:
            add = len(w) + (1 if cur else 0)
            if cur_len + add <= max_length:
                cur.append(w); cur_len += add
            else:
                lines.append(" ".join(cur))
                cur = [w]; cur_len = len(w)
        if cur:
            lines.append(" ".join(cur))
        return "\n".join(lines)

    # Mapeamento: qual “bloco” de espécie vai para quais linhas
    # bloco 0 -> rows (0,1)
    # bloco 1 -> rows (3,4)
    # bloco 2 -> rows (6,7)
    block_to_rows = [(0, 1), (3, 4), (6, 7)]

    species_idx = 0
    for j in range(n_cols):
        for block in range(species_per_col):
            r0, r1 = block_to_rows[block]

            if species_idx >= n_species:
                axes[r0, j].axis("off")
                axes[r1, j].axis("off")
                continue

            species = unique_species[species_idx]
            species_idx += 1

            samples = list(species_samples[species])
            while len(samples) < samples_per_species:
                samples.append(np.zeros((100, 100, 3), dtype=np.uint8))

            axes[r0, j].imshow(samples[0])
            axes[r1, j].imshow(samples[1])

            title = break_long_title(species)
            axes[r0, j].text(0.5, 1.08, title, transform=axes[r0, j].transAxes,
                             ha="center", va="bottom", fontsize=12)

    # Spacer rows sempre off
    for r in spacer_rows:
        for j in range(n_cols):
            axes[r, j].axis("off")

    fig.subplots_adjust(top=0.95, bottom=0.03, left=0.02, right=0.98)

    plt.savefig(f"{dataset_name}_species_grid_3percol.pdf", bbox_inches="tight", dpi=100)
    plt.show()


In [ ]:
# Plot com 3 espécies por coluna (6 linhas)
plot_species_grid_3percol(amazon_samples, "Amazon Data")

## 2. Bioflore Data

Arquivos utilizados:
- Shapes: `/home/luizluz/Documentos/multi-task-fcn/bioflore_data/shapes/labels.shp`
- Imagens RGB: `/home/luizluz/Documentos/multi-task-fcn/matematica_industria_data/raw/geotiffs/`

Vamos iterar sobre os shapes, identificar a espécie, recortar do mosaico RGB correspondente e aplicar a máscara.

In [ ]:
bioflore_shp_path = '/home/luizluz/Documentos/multi-task-fcn/bioflore_data/shapes/labels.shp'
bioflore_rgb_dir = '/home/luizluz/Documentos/multi-task-fcn/matematica_industria_data/raw/geotiffs'

# Ler shapefile
gdf = gpd.read_file(bioflore_shp_path)
print(f"Colunas no shapefile: {gdf.columns}")

# Mapeamento de mosaicos RGB
rgb_mosaics = {
    'Mosaic_BigPlot_03.tif': os.path.join(bioflore_rgb_dir, 'Mosaic_BigPlot_03.tif'),
    'Mosaic_BigPlot_07.tif': os.path.join(bioflore_rgb_dir, 'Mosaic_BigPlot_07.tif'),
    'Mosaic_BigPlot_11.tif': os.path.join(bioflore_rgb_dir, 'Mosaic_BigPlot_11.tif')
}

# Usar coluna 'species'
sp_col = 'species'
unique_species_bio = gdf[sp_col].unique()
print(f"Espécies Bioflore: {unique_species_bio}")

bioflore_samples = {sp: [] for sp in unique_species_bio}

def extract_bioflore_samples(gdf, rgb_mosaics, samples_dict, sp_column, max_samples=4):
    # Abrir os mosaicos RGB
    open_mosaics = {}
    for k, v in rgb_mosaics.items():
        if os.path.exists(v):
            open_mosaics[k] = rasterio.open(v)
            print(f"Aberto: {k} com {open_mosaics[k].count} bandas")
        else:
            print(f"Aviso: Mosaico {v} não encontrado.")
    
    if not open_mosaics:
        print("Nenhum mosaico encontrado.")
        return

    for species in samples_dict.keys():
        print(f"Processando Bioflore: {species}")
        
        # Filtrar geometrias dessa espécie
        subset = gdf[gdf[sp_column] == species]
        count = 0
        # Lista para guardar os patches já processados (py, px, size)
        processed_patches = []
        
        for idx, row in subset.iterrows():
            if count >= max_samples:
                break
            
            geom = row.geometry
            tree_id = row['id']
            mosaic_name = row['geotiff']
            
            # Verificar se temos esse mosaico aberto
            if mosaic_name not in open_mosaics:
                continue
            
            src = open_mosaics[mosaic_name]
            
            try:
                # Pegar bounds da geometria
                minx, miny, maxx, maxy = geom.bounds
                center_x = (minx + maxx) / 2
                center_y = (miny + maxy) / 2
                
                # Converter para índices de pixel
                py, px = src.index(center_x, center_y)
                
                # Verificar se está dentro do mosaico
                if not (0 <= py < src.height and 0 <= px < src.width):
                    continue
                
                # Calcular tamanho baseado na geometria
                width_m = maxx - minx
                height_m = maxy - miny
                
                # Converter para pixels (usando resolução do raster)
                res_x, res_y = src.res
                width_px = int(width_m / abs(res_x) * 1.05)  # 50% margem
                height_px = int(height_m / abs(res_y) * 1.05)
                
                # Tamanho quadrado
                size = max(100, max(width_px, height_px))
                if size % 2 != 0:
                    size += 1
                
                half = size // 2
                
                # Verificar limites
                if (px - half < 0 or px + half > src.width or
                    py - half < 0 or py + half > src.height):
                    continue
                
                # Verificar se este patch sobrepõe com algum já processado
                overlaps = False
                for prev_py, prev_px, prev_size in processed_patches:
                    if patches_overlap(py, px, size, prev_py, prev_px, prev_size):
                        overlaps = True
                        break
                
                if overlaps:
                    continue  # Pular esta árvore se sobrepõe com uma já processada
                
                window = Window(px - half, py - half, size, size)
                
                # Ler RGB (primeiras 3 bandas)
                rgb_crop = src.read([1, 2, 3], window=window)
                
                # Verificar se leu dados válidos
                if rgb_crop.max() == 0:
                    continue
                
                # Criar máscara usando rasterio.features.geometry_mask
                win_transform = src.window_transform(window)
                
                # geometry_mask: True = fora, False = dentro (invert=True inverte)
                mask = rasterio.features.geometry_mask(
                    [geom],
                    out_shape=(size, size),
                    transform=win_transform,
                    invert=True
                )
                
                # Aplicar máscara (fundo preto)
                rgb_masked = rgb_crop.copy()
                for b in range(3):
                    rgb_masked[b][~mask] = 0
                
                # Transpor para (H, W, 3)
                img_show = np.transpose(rgb_masked, (1, 2, 0))
                
                # Adicionar à lista
                samples_dict[species].append(img_show)
                # Registrar este patch como processado
                processed_patches.append((py, px, size))
                count += 1
                
            except Exception as e:
                print(f"Erro processando árvore {tree_id}: {e}")
                continue

    # Fechar arquivos
    for src in open_mosaics.values():
        src.close()

extract_bioflore_samples(gdf, rgb_mosaics, bioflore_samples, sp_col)


In [ ]:

# plot_species_grid(bioflore_samples, "Bioflore Data")

plot_species_grid_wrapped(bioflore_samples, "Bioflore Data")
